In [ ]:
cutting_speed_mm_min = 1000 # mm/min
offset_general_s = 120 #s 
offset_time_smartgate_s = 2 #s
offset_time_sortmaster_s = 5 #s
offset_time_manually_s = 20 #s
offset_time_hole_s = 3 #s

smartgate_min = (30, 30) # min
smartgate_max = (160, 160) # max

sortmaster_min = (60, 90) # min
sortmaster_max = (1500, 2000) # max

def get_cutting_time(path):
    cutting_time_s = get_cutting_length(path) / (cutting_speed_mm_min / 60)

    extraction_method = get_extraction_method(path)

    if extraction_method == "Gate + Master":
        offset_extraction_method_s = min(offset_time_smartgate_s, offset_time_sortmaster_s)
    elif extraction_method == "Master":
        offset_extraction_method_s = offset_time_sortmaster_s
    elif extraction_method == "Gate":
        offset_extraction_method_s = offset_time_smartgate_s
    else:
        offset_extraction_method_s = offset_time_manually_s

    offset_time_total_s = offset_general_s + offset_extraction_method_s + get_number_of_holes(path) * offset_time_hole_s

    return cutting_time_s + offset_time_total_s

def get_fit(object, min_opening, max_opening):
    ob_1, ob_2 = sorted(object)
    min_1, min_2 = sorted(min_opening)
    max_1, max_2 = sorted(max_opening)

    return min_1 <= ob_1 <= max_1 and min_2 <= ob_2 <= max_2

def get_extraction_method(path):

    bounding_box = tuple(sorted(get_dxf_bound(path)))

    if get_fit(bounding_box, sortmaster_min, sortmaster_max) and get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate + Master"
    
    elif get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate"
    
    elif get_fit(bounding_box, sortmaster_min, sortmaster_max):
        return "Master"
    
    else:
        return "Manual"

def get_bounding_box_area(path):
    bounding_box = get_dxf_bound(path)
    return bounding_box[0] * bounding_box[1]

def get_automation_index(extraction_method):

    if extraction_method != "Manual":
        return 1
    else:
        return 0

csv_df['path'] = folder + csv_df['file_name']

csv_df[['bounding_box_width_mm', 'bounding_box_height_mm']] = csv_df['path'].apply(lambda p: pd.Series(get_dxf_bound(p)))

csv_df['bounding_box_area_mm^2'] = csv_df['bounding_box_width_mm'] * csv_df['bounding_box_height_mm']
csv_df['part_area_mm^2'] = csv_df['path'].apply(get_part_area)
csv_df['aspect_ratio'] = csv_df['bounding_box_height_mm'] / csv_df['bounding_box_width_mm']
csv_df['rectangularity_score'] = csv_df['part_area_mm^2'] / csv_df['bounding_box_area_mm^2']

csv_df['extraction_method'] = csv_df["path"].apply(get_extraction_method)
csv_df['automation_index'] = csv_df["extraction_method"].map(get_automation_index)

csv_df['number_of_holes'] = csv_df["path"].apply(get_number_of_holes)
csv_df['cutting_length_mm'] = csv_df["path"].apply(get_cutting_length)
csv_df['cutting_time_s'] = csv_df["path"].apply(get_cutting_time)